# 07 — Foreign Flow Mechanism Discovery V1

**Phase A: descriptive / outcome-blind data forensics**

Research question tahap ini:

> Seperti apa anatomi foreign participation di IDX: seberapa persistent, seberapa sering ekstrem, saham seperti apa yang menerima accumulation/distribution kuat, dan bagaimana slow-state vs fast-state foreign flow terbentuk?

Notebook ini **belum** menghitung future return, IC, Sharpe, TP/SL outcome, atau fit model. Tujuannya supaya kita melihat datanya sendiri sebelum merumuskan hypothesis predictive.

## Cara pakai

1. Jalankan cell dari atas ke bawah.
2. `REP_PATH` sudah diarahkan ke artifact Foreign Flow V2 historis yang pernah kita materialize.
3. `RAW_FLOW_PATH` opsional; isi kalau path raw archive sudah diketahui.
4. Ubah `DRILL_TICKER` untuk membedah ticker tertentu.
5. Jangan arahkan notebook ini ke O2/protected/fresh-forward outcome artifact.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

REP_PATH = Path(r'D:\Documents\Project\idx-trade-foreign-flow-representation-v2-20260815-001\foreign_flow_representation_v2.parquet')
RAW_FLOW_PATH = None  # optional: Path(r'...')
DRILL_TICKER = 'BBCA'
SAVE_FIGURES = False
FIG_DIR = Path('notebooks/_figures/foreign_flow_mechanism_discovery_v1')
if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path) if path.suffix.lower() == '.parquet' else pd.read_csv(path)

def finish_plot(name=None):
    plt.tight_layout()
    if SAVE_FIGURES and name:
        plt.savefig(FIG_DIR / f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()

## 1. Load + census

Representation V2 bukan jawaban final; ini hanya starting point yang sudah causally hardened. Pertama lihat identity, coverage, dan kolom yang benar-benar ada.

In [ ]:
rep = read_table(REP_PATH).copy()
required = {'ticker','feature_session','flow_through_session'}
miss = required - set(rep.columns)
if miss:
    raise KeyError(f'Missing required columns: {sorted(miss)}')
rep['ticker'] = rep['ticker'].astype(str).str.upper().str.strip()
rep['feature_session'] = pd.to_datetime(rep['feature_session'], errors='raise')
rep['flow_through_session'] = pd.to_datetime(rep['flow_through_session'], errors='raise')
if rep.duplicated(['ticker','feature_session']).any():
    raise ValueError('Duplicate ticker/feature_session rows detected')
feature_cols = [c for c in rep.columns if c not in required]
print('shape:', rep.shape)
print('tickers:', rep['ticker'].nunique())
print('sessions:', rep['feature_session'].nunique())
print('range:', rep['feature_session'].min().date(), '->', rep['feature_session'].max().date())
display(pd.DataFrame({'feature': feature_cols}))

## 2. Coverage through time

Kalau coverage berubah besar, perubahan distribution signal belum tentu perubahan behavior market.

In [ ]:
coverage = rep.groupby('feature_session').agg(rows=('ticker','size'), tickers=('ticker','nunique')).reset_index().sort_values('feature_session')
display(coverage.describe(include='all'))
plt.figure(figsize=(11,4))
plt.plot(coverage['feature_session'], coverage['tickers'])
plt.title('Foreign Flow representation — ticker coverage per session')
plt.xlabel('Feature session'); plt.ylabel('Unique tickers')
finish_plot('01_ticker_coverage')

## 3. Missingness

Missingness bisa berasal dari warm-up, universe restriction, listing state, atau source availability. Jangan otomatis menganggap semuanya data rusak.

In [ ]:
missingness = rep[feature_cols].isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
missingness['missing_pct'] = 100 * missingness['missing_rate']
display(missingness)
plt.figure(figsize=(10, max(4, .32*len(missingness))))
plt.barh(missingness.index[::-1], missingness['missing_pct'].iloc[::-1])
plt.title('Missingness by Foreign Flow feature'); plt.xlabel('Missing observations (%)')
finish_plot('02_feature_missingness')

## 4. Distribution signal utama

Kita lihat participation, abnormal shock, own-history percentile, persistence, dan acceleration. Histogram membatasi axis visual ke P1–P99 saja; raw observation tidak dihapus atau diubah.

In [ ]:
inspect = [c for c in ['foreign_participation_1','foreign_flow_shock_1','foreign_flow_shock_percentile_120','foreign_weighted_persistence_5','foreign_weighted_persistence_20','foreign_flow_acceleration_5_20'] if c in rep.columns]
display(rep[inspect].describe(percentiles=[.001,.01,.05,.25,.5,.75,.95,.99,.999]).T)
for col in inspect:
    x = rep[col].dropna()
    if x.empty:
        continue
    lo, hi = x.quantile([.01,.99])
    xp = x[(x >= lo) & (x <= hi)]
    plt.figure(figsize=(9,3.6))
    plt.hist(xp, bins=70)
    plt.title(f'{col} — central 98% for visualization')
    plt.xlabel(col); plt.ylabel('Observations')
    finish_plot(f'03_dist_{col}')

## 5. Extreme observations

Audit lama menemukan cluster ekstrem. Kali ini kita tampilkan dan investigasi; jangan langsung winsorize. Pertanyaan: genuine flow, denominator artifact, illiquidity, corporate action, atau perubahan listing/tradability?

In [ ]:
extreme_cols = [c for c in ['ticker','flow_through_session','feature_session','foreign_flow_shock_1','foreign_participation_1','foreign_flow_shock_percentile_120','foreign_weighted_persistence_20','foreign_flow_acceleration_5_20'] if c in rep.columns]
print('MOST EXTREME POSITIVE')
display(rep.nlargest(25, 'foreign_flow_shock_1')[extreme_cols])
print('MOST EXTREME NEGATIVE')
display(rep.nsmallest(25, 'foreign_flow_shock_1')[extreme_cols])
extreme_abs = rep.loc[rep['foreign_flow_shock_1'].notna(), ['ticker','foreign_flow_shock_1']].assign(abs_shock=lambda x: x['foreign_flow_shock_1'].abs()).nlargest(250,'abs_shock')
repeat_extreme = extreme_abs.groupby('ticker').agg(extreme_events=('ticker','size'), max_abs_shock=('abs_shock','max')).sort_values(['extreme_events','max_abs_shock'], ascending=False)
display(repeat_extreme.head(30))
plot_repeat = repeat_extreme.head(20).sort_values('extreme_events')
plt.figure(figsize=(10,5)); plt.barh(plot_repeat.index, plot_repeat['extreme_events'])
plt.title('Repeated presence among top-250 absolute flow shocks'); plt.xlabel('Extreme observations')
finish_plot('04_repeated_extreme_tickers')

## 6. Short state vs medium state

Kalau persistence 5 dan 20 selalu sama, dua timescale kita redundant. Kalau banyak titik jauh dari diagonal, recent flow sedang berbeda dari medium-term state.

In [ ]:
needed = {'foreign_weighted_persistence_5','foreign_weighted_persistence_20'}
if needed.issubset(rep.columns):
    p = rep[list(needed)].dropna()
    if len(p) > 100000:
        p = p.sample(100000, random_state=42)
    plt.figure(figsize=(6,6))
    plt.scatter(p['foreign_weighted_persistence_20'], p['foreign_weighted_persistence_5'], s=4, alpha=.12)
    plt.axhline(0, linewidth=1); plt.axvline(0, linewidth=1); plt.plot([-1,1],[-1,1], linestyle='--', linewidth=1)
    plt.xlim(-1.02,1.02); plt.ylim(-1.02,1.02)
    plt.title('Short vs medium Foreign Flow persistence')
    plt.xlabel('20-session persistence'); plt.ylabel('5-session persistence')
    finish_plot('05_persistence_5_vs_20')

## 7. Ticker drill-down

Ubah `DRILL_TICKER` di CONFIG. Tujuannya melihat satu saham dengan mata sendiri: noise, slow accumulation state, regime shift, atau campuran semuanya.

In [ ]:
one = rep.loc[rep['ticker'].eq(DRILL_TICKER)].sort_values('feature_session').copy()
if one.empty:
    raise ValueError(f'{DRILL_TICKER} not found')
print(DRILL_TICKER, one['feature_session'].min().date(), '->', one['feature_session'].max().date(), 'rows:', len(one))
for col in ['foreign_flow_shock_percentile_120','foreign_weighted_persistence_20','foreign_flow_acceleration_5_20']:
    if col not in one.columns:
        continue
    y = one[col].copy()
    if col == 'foreign_flow_acceleration_5_20':
        lo, hi = y.quantile([.01,.99]); y = y.clip(lower=lo, upper=hi)
    plt.figure(figsize=(12,3.4))
    plt.plot(one['feature_session'], y)
    plt.axhline(0 if col != 'foreign_flow_shock_percentile_120' else .5, linewidth=1)
    plt.title(f'{DRILL_TICKER} — {col}')
    finish_plot(f'06_{DRILL_TICKER}_{col}')

## 8. OPTIONAL — long signal memory dari raw Foreign Flow

Kalau `RAW_FLOW_PATH` diisi, kita hitung exploratory directional balance 5/20/60/120/250 **archive sessions**: `sum(net) / sum(buy + sell)`. Sebelum rolling, setiap ticker direindex ke global session grid yang ada di accepted raw archive; missing ticker-session tetap `NaN`, bukan dianggap zero.

Ini **bukan accepted alpha feature**; hanya descriptive lens bounded `[-1,+1]` untuk melihat slow accumulation/distribution state.

In [ ]:
raw = None
if RAW_FLOW_PATH is None:
    print('RAW_FLOW_PATH belum diisi — long-memory section dilewati.')
else:
    raw = read_table(RAW_FLOW_PATH).copy()
    req = {'ticker','session_date','foreign_buy','foreign_sell','foreign_net','unit'}
    miss = req - set(raw.columns)
    if miss:
        raise KeyError(f'Missing raw columns: {sorted(miss)}')
    raw['ticker'] = raw['ticker'].astype(str).str.upper().str.strip()
    raw['session_date'] = pd.to_datetime(raw['session_date'], errors='raise').dt.normalize()
    for c in ['foreign_buy','foreign_sell','foreign_net']:
        raw[c] = pd.to_numeric(raw[c], errors='raise')
    if raw.duplicated(['ticker','session_date']).any():
        raise ValueError('Duplicate raw ticker/session rows')
    if not raw['foreign_net'].eq(raw['foreign_buy'] - raw['foreign_sell']).all():
        raise ValueError('foreign_net identity mismatch')
    if not raw['unit'].astype(str).eq('SHARES').all():
        raise ValueError('Expected unit SHARES')
    raw = raw.sort_values(['ticker','session_date']).reset_index(drop=True)
    raw['foreign_gross'] = raw['foreign_buy'] + raw['foreign_sell']
    archive_sessions = pd.DatetimeIndex(sorted(raw['session_date'].unique()))
    windows = [5,20,60,120,250]
    pieces = []
    for ticker, g in raw.groupby('ticker', sort=False):
        g = g.set_index('session_date').reindex(archive_sessions)
        g.index.name = 'session_date'
        g['ticker'] = ticker
        for w in windows:
            net = g['foreign_net'].rolling(w, min_periods=w).sum()
            gross = g['foreign_gross'].rolling(w, min_periods=w).sum()
            g[f'directional_balance_{w}'] = np.where(gross > 0, net/gross, np.nan)
        pieces.append(g.reset_index())
    raw_state = pd.concat(pieces, ignore_index=True)
    t = raw_state.loc[raw_state['ticker'].eq(DRILL_TICKER)].copy()
    for w in windows:
        plt.figure(figsize=(12,3.2))
        plt.plot(t['session_date'], t[f'directional_balance_{w}'])
        plt.axhline(0, linewidth=1); plt.ylim(-1.02,1.02)
        plt.title(f'{DRILL_TICKER} — foreign directional balance, {w} archive sessions')
        finish_plot(f'07_{DRILL_TICKER}_directional_balance_{w}')

## 9. Research notes — isi setelah lihat chart

Jangan cari chart paling bullish. Catat fenomena yang benar-benar terlihat.

1. Coverage stabil atau ada structural break?
2. Feature mana paling sparse?
3. Distribution foreign pressure simetris atau sangat skewed?
4. Extreme shocks didominasi ticker tertentu atau tersebar?
5. Extreme cluster terlihat seperti data-quality problem atau plausible market event?
6. Persistence 5 membawa state berbeda dari persistence 20?
7. Pada ticker yang kamu kenal, apakah Foreign Flow menunjukkan slow state yang masuk akal?
8. Kalau raw section tersedia, apakah 60/120/250-session balance benar-benar berbeda dari 5/20-session balance?
9. Fenomena apa yang paling aneh dan layak dibedah?

**Stop di sini dulu.** Phase B baru membuka historical future returns `H1/H3/H5/H10/H20` setelah kita menulis mechanism hypotheses dari Phase A.